# **Notebook 7: Solution V2 — Fine-Tuned RAG Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `./intent_lora_best/` — Created by Notebook 6
- [ ] `./chroma_db/` — Created by Notebook 4
- [ ] `df_test.csv` — Created by Notebook 2
- [ ] `outputs.json` + `v1_metrics.csv` — From Notebooks 3/4/5
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `Comparative_Results_Full.csv` + `Comparative_Results_Summary.csv` _(Final deliverables)_

---

### **Task 4.3: Integrate Fine-Tuned Model with Retrieval**

#### **4.3.1 Integrate Fine-Tuned Model into Existing RAG Pipeline [3 marks]**
**The Task:** Replace the baseline model with the fine-tuned model acting as an intent router. Merge the LoRA adapters, extract a JSON intent, map it to a vector-search string, retrieve, and generate. Validate the integrated system.

**Hints & Tips:**
* `PeftModel.from_pretrained(base_model, "./intent_lora_best").merge_and_unload()` fuses the adapters for fast inference.
* Use a strong system prompt with few-shot examples so the router emits JSON only; `re.search(r'\{.*?\}', raw)` is a safety net for stray preamble.
* Map the intent (e.g. `track_order`) to an SOP header search string (e.g. `# Track Order`). Fall back to the raw query if JSON parsing fails.
* Validate end-to-end on `test_query`: intent → search string → retrieved SOP → final answer.

**Parameter Tuning:**
* `max_new_tokens=30` for the router (JSON is short — more tokens invite trailing explanation text).
* 4 few-shot examples is the sweet spot.

**Learner Inference:** Querying with the structured intent keyword instead of the noisy prompt retrieves the exact policy clause — the core of Hybrid RAG.

In [1]:

import torch
import json
import re
import pandas as pd

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from sentence_transformers import SentenceTransformer
import chromadb

# --- Config (locked across ALL notebooks) ---
MODEL_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
EMBED_ID   = "sentence-transformers/all-MiniLM-L6-v2"
ADAPTER_DIR = "./intent_lora_best"
CHROMA_DIR  = "./chroma_db"

# --- Device: Mac GPU (MPS), else CPU ---
device = "mps" if torch.backends.mps.is_available() else "cpu"

print("Device:", device)

/Users/abinas/Documents/projects/machine-learning/capstone-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [6]:
# YOUR CODE HERE
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
).to(device)

router = PeftModel.from_pretrained(base_model, ADAPTER_DIR)

router.eval()
print("Router loaded (adapter toggleable). Ready for inference.")

Loading weights: 100%|██████████| 338/338 [00:04<00:00, 81.51it/s] 


Router loaded (adapter toggleable). Ready for inference.


In [7]:
# YOUR CODE HERE
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBED_ID)
vectordb   = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)

def _run_model(messages):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        with router.disable_adapter():
            out = router.generate(**inputs, max_new_tokens=256, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def extract_intent(query):
    messages = [{"role": "user", "content": query}]          
    raw = _run_model_short(messages)                         
    match = re.search(r'\{.*?\}', raw, re.DOTALL)            
    if match:
        try:
            return json.loads(match.group())                
        except json.JSONDecodeError:
            pass
    return None                                             

def _run_model_short(messages):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = router.generate(**inputs, max_new_tokens=30, do_sample=False)
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def hybrid_rag(query):
    intent_json = extract_intent(query)

    if intent_json and "intent" in intent_json:
        search_string = intent_json["intent"].replace("_", " ")   
    else:
        search_string = query                                     

    top_doc = vectordb.similarity_search(search_string, k=1)[0].page_content

    messages = [
        {"role": "system", "content": f"Answer strictly using this SOP:\n{top_doc}"},
        {"role": "user", "content": query},
    ]
    answer = _run_model(messages)
    return {"intent_json": intent_json, "search_string": search_string,
            "retrieved_sop": top_doc, "answer": answer}

print("Hybrid RAG pipeline ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6171.71it/s]


Hybrid RAG pipeline ready.


In [8]:
test_query = "ugh my order still hasn't shown up and I'm getting frustrated"

result = hybrid_rag(test_query)

print("Raw query      :", test_query)
print("Router JSON    :", result["intent_json"])
print("Search string  :", result["search_string"])
print("Retrieved SOP  :", result["retrieved_sop"][:60], "...")
print("\n--- Final Answer ---")
print(result["answer"])

Raw query      : ugh my order still hasn't shown up and I'm getting frustrated
Router JSON    : {'intent': 'track_order', 'category': 'ORDER'}
Search string  : track order
Retrieved SOP  : # Order Tracking

## Scope
Helps customers locate an order a ...

--- Final Answer ---
I understand you're feeling frustrated. Let's try these steps:

1. **Check Your Email**: Sometimes orders don’t get scanned immediately. Check your inbox again within the next 24 hours.

2. **Contact Customer Service**: Reach out to our support team via the provided contact information. They can help expedite the process if there are any issues.

3. **Carrier Investigation**: If the order is still not showing up, we might need to investigate further. We’ll reach out to the carrier directly to ensure they’re scanning correctly.

Please let us know how things progress!


### **Task 4.4: Evaluate Solution V2**

#### **4.4.1 Re-Execute Evaluation Framework [3 marks]**
**The Task:** Evaluate Format Adherence and Intent Accuracy on the held-out test split (zero leakage guaranteed) and an adversarial subset derived via regex filtering. Evaluate the final synthesis using ROUGE/BLEU.

**Hints & Tips:**
* Reuse `df_test` from Notebook 2 — it's the leakage-free test split.
* Build the adversarial subset by regex-filtering for sentiment/hedging words (`still`, `never`, `terrible`, `frustrated`).
* Report Format Adherence %, Exact Match %, and Fuzzy Match % (fuzzy catches `order_tracking` vs `track_order`).

**Learner Inference:** Using the held-out test split guarantees zero leakage and trustworthy scores.

In [10]:
# YOUR CODE HERE
from tqdm import tqdm

df_test = pd.read_csv("df_test.csv")
adversarial_pattern = r"(?:do not know|don't know|not sure|how do i|how can i|unable|cannot|can not|trying to)"
df_adversarial = df_test[
    df_test["instruction"].str.contains(adversarial_pattern, case=False, regex=True)
].reset_index(drop=True)

print(f"Full test rows   : {len(df_test)}")
print(f"Adversarial rows : {len(df_adversarial)}")

N = 50
df_eval = df_test.sample(min(N, len(df_test)), random_state=42).reset_index(drop=True)

def run_router_over(df):
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        rows.append({
            "query": row["instruction"],
            "true_intent": row["intent"],
            "pred_json": extract_intent(row["instruction"]),
        })
    return rows

print("\nRunning router on full-test sample...")
eval_results = run_router_over(df_eval)

print("Running router on adversarial subset...")
adv_results = run_router_over(df_adversarial)

print(f"\nCollected: {len(eval_results)} eval rows, {len(adv_results)} adversarial rows")

Full test rows   : 391
Adversarial rows : 70

Running router on full-test sample...


100%|██████████| 50/50 [00:36<00:00,  1.35it/s]


Running router on adversarial subset...


100%|██████████| 70/70 [00:49<00:00,  1.40it/s]


Collected: 50 eval rows, 70 adversarial rows


In [11]:
def normalize(s):
    """Lowercase, underscores->spaces, split into a word-set (order-independent)."""
    return set(str(s).lower().replace("_", " ").split())

def score_metrics(results):
    total = len(results)
    format_ok = exact_ok = fuzzy_ok = 0
    for r in results:
        pred, true = r["pred_json"], r["true_intent"]
        if isinstance(pred, dict) and "intent" in pred:  
            format_ok += 1
            pi = pred["intent"]
            if pi == true:                                  
                exact_ok += 1
            if normalize(pi) == normalize(true):            
                fuzzy_ok += 1
    return {
        "Format Adherence %": format_ok / total * 100,
        "Exact Match %":      exact_ok  / total * 100,
        "Fuzzy Match %":      fuzzy_ok  / total * 100,
    }

v2_full = score_metrics(eval_results)
v2_adv  = score_metrics(adv_results)

print("=== Solution V2 (Hybrid RAG) — Router Metrics ===\n")
print(f"{'Metric':22s} | {'Full Test':>10s} | {'Adversarial':>11s}")
print("-" * 50)
for m in v2_full:
    print(f"{m:22s} | {v2_full[m]:9.1f}% | {v2_adv[m]:10.1f}%")

=== Solution V2 (Hybrid RAG) — Router Metrics ===

Metric                 |  Full Test | Adversarial
--------------------------------------------------
Format Adherence %     |     100.0% |      100.0%
Exact Match %          |      70.0% |       74.3%
Fuzzy Match %          |      70.0% |       74.3%


#### **4.4.2 Analyse Fine-Tuning Impact [2 marks]**
**The Task:** Compare Solution V1 (Naive RAG) against Solution V2 (Hybrid RAG) to quantify the improvement attributable to fine-tuning.

**Hints & Tips:**
* Load `v1_metrics.csv` from Notebook 5 and compare against the V2 scores you just computed.
* Compute improvement percentages: `(v2 - v1) / v1 * 100` for each metric.
* Attribute the delta specifically to fine-tuning — retrieval was already present in V1, so any gain here is the router's contribution.

**Learner Inference:** This isolates fine-tuning's contribution, just as Task 3.4 isolated retrieval's — together they decompose the full system's improvement.

In [12]:
# YOUR CODE HERE

v1_metrics = pd.read_csv("v1_metrics.csv")
print("V1 metrics (from NB5):")
print(v1_metrics.to_string(index=False))

def v1_value(metric_name):
    row = v1_metrics[v1_metrics["Metric"].str.contains(metric_name, case=False)]
    return float(row["Naive_RAG"].iloc[0]) if len(row) else float("nan")

v1_format = v1_value("Format Adherence")
v2_format = v2_full["Format Adherence %"]

def improvement(v1, v2):
    return (v2 - v1) / v1 * 100 if v1 else float("nan")

print("\n=== Fine-Tuning Impact: V1 (Naive RAG) → V2 (Hybrid RAG) ===\n")
print(f"{'Metric':24s} | {'V1':>8s} | {'V2':>8s} | {'Change':>10s}")
print("-" * 60)
print(f"{'Format Adherence %':24s} | {v1_format:7.1f}% | {v2_format:7.1f}% | {improvement(v1_format, v2_format):+9.1f}%")
print(f"{'Intent Acc (Exact) %':24s} | {'n/a':>8s} | {v2_full['Exact Match %']:7.1f}% | {'NEW in V2':>10s}")
print(f"{'Intent Acc (Fuzzy) %':24s} | {'n/a':>8s} | {v2_full['Fuzzy Match %']:7.1f}% | {'NEW in V2':>10s}")

print("\nInterpretation:")
print("- V1 had NO structured intent extraction; the fine-tuned router adds it.")
print("- Intent Accuracy (70%+) is a capability that only exists from V2 onward.")
print("- This isolates fine-tuning's contribution: retrieval was already in V1,")
print("  so the router's intent extraction is the new value fine-tuning delivers.")

V1 metrics (from NB5):
            Metric   Baseline  Naive_RAG  Improvement_%
  Format Adherence 100.000000 100.000000       0.000000
           ROUGE-1   0.177233   0.191580       8.094629
           ROUGE-L   0.081020   0.103350      27.560016
              BLEU   0.002000   0.006404     220.188895
       Consistency 100.000000 100.000000       0.000000
Hallucination Freq  34.000000  28.000000      17.647059

=== Fine-Tuning Impact: V1 (Naive RAG) → V2 (Hybrid RAG) ===

Metric                   |       V1 |       V2 |     Change
------------------------------------------------------------
Format Adherence %       |   100.0% |   100.0% |      +0.0%
Intent Acc (Exact) %     |      n/a |    70.0% |  NEW in V2
Intent Acc (Fuzzy) %     |      n/a |    70.0% |  NEW in V2

Interpretation:
- V1 had NO structured intent extraction; the fine-tuned router adds it.
- Intent Accuracy (70%+) is a capability that only exists from V2 onward.
- This isolates fine-tuning's contribution: retrieval was

### **Task 4.5: Perform Comparative Analysis**

> Subtasks 4.5.1 (Compare All Versions) and 4.5.2 (Document Findings) are written up in the **Comparative Analysis Report PDF**. The cell below generates the scoring tables that feed that report.

**The Task:** Run all three architectures (Baseline, Naive RAG, Hybrid RAG) across the full held-out test split with SOP-grounded references, then export the per-row and summary CSVs.

**Hints & Tips:**
* SOP-grounded references reward policy-specific answers, ensuring Hybrid scores highest.
* This is the most compute-intensive cell — expect 15–30 min on T4. Use `df_test.head(50)` if time-constrained.
* Export `Comparative_Results_Full.csv` (per-row) and `Comparative_Results_Summary.csv` (aggregate).

In [13]:
%pip install -q rouge-score nltk

Note: you may need to restart the kernel to use updated packages.


In [14]:
# YOUR CODE HERE
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
smooth = SmoothingFunction().method1

def generate_baseline(query):
    messages = [{"role": "user", "content": query}]
    return _run_model(messages)

def generate_naive_rag(query):
    top_doc = vectordb.similarity_search(query, k=1)[0].page_content
    messages = [
        {"role": "system", "content": f"Answer strictly using this SOP:\n{top_doc}"},
        {"role": "user", "content": query},
    ]
    return _run_model(messages)

def score_answer(prediction, reference):
    r = scorer.score(reference, prediction)
    bleu = sentence_bleu([reference.split()], prediction.split(), smoothing_function=smooth)
    return r["rouge1"].fmeasure, r["rougeL"].fmeasure, bleu

SAMPLE = 30
df_cmp = df_test.sample(min(SAMPLE, len(df_test)), random_state=42).reset_index(drop=True)

full_rows = []
for _, row in tqdm(df_cmp.iterrows(), total=len(df_cmp)):
    q = row["instruction"]
    hy  = hybrid_rag(q)
    sop = hy["retrieved_sop"]                    # SOP-grounded reference

    base_ans, naive_ans, hyb_ans = generate_baseline(q), generate_naive_rag(q), hy["answer"]

    b = score_answer(base_ans,  sop)
    n = score_answer(naive_ans, sop)
    h = score_answer(hyb_ans,   sop)

    full_rows.append({
        "query": q, "true_intent": row["intent"], "reference_sop": sop[:200],
        "baseline_answer": base_ans, "naive_answer": naive_ans, "hybrid_answer": hyb_ans,
        "baseline_rouge1": b[0], "baseline_rougeL": b[1], "baseline_bleu": b[2],
        "naive_rouge1": n[0],    "naive_rougeL": n[1],    "naive_bleu": n[2],
        "hybrid_rouge1": h[0],   "hybrid_rougeL": h[1],   "hybrid_bleu": h[2],
    })

df_full = pd.DataFrame(full_rows)
print("Scored", len(df_full), "rows across all 3 architectures.")

100%|██████████| 30/30 [08:50<00:00, 17.67s/it]

Scored 30 rows across all 3 architectures.


In [15]:
# YOUR CODE HERE

df_full.to_csv("Comparative_Results_Full.csv", index=False)
print("Saved Comparative_Results_Full.csv  (", len(df_full), "rows )")

summary = pd.DataFrame({
    "Architecture": ["Baseline", "Naive_RAG", "Hybrid_RAG"],
    "ROUGE-1": [df_full["baseline_rouge1"].mean(), df_full["naive_rouge1"].mean(), df_full["hybrid_rouge1"].mean()],
    "ROUGE-L": [df_full["baseline_rougeL"].mean(), df_full["naive_rougeL"].mean(), df_full["hybrid_rougeL"].mean()],
    "BLEU":    [df_full["baseline_bleu"].mean(),   df_full["naive_bleu"].mean(),   df_full["hybrid_bleu"].mean()],
})

summary["Intent_Exact_%"] = ["n/a", "n/a", round(v2_full["Exact Match %"], 1)]
summary["Intent_Fuzzy_%"] = ["n/a", "n/a", round(v2_full["Fuzzy Match %"], 1)]
summary["Format_%"]       = ["n/a", "n/a", round(v2_full["Format Adherence %"], 1)]

summary.to_csv("Comparative_Results_Summary.csv", index=False)
print("Saved Comparative_Results_Summary.csv\n")
print(summary.to_string(index=False))

Saved Comparative_Results_Full.csv  ( 30 rows )
Saved Comparative_Results_Summary.csv

Architecture  ROUGE-1  ROUGE-L     BLEU Intent_Exact_% Intent_Fuzzy_% Format_%
    Baseline 0.165189 0.077928 0.002001            n/a            n/a      n/a
   Naive_RAG 0.181585 0.090107 0.003467            n/a            n/a      n/a
  Hybrid_RAG 0.160460 0.082990 0.003359           70.0           70.0    100.0


---
## END-OF-NOTEBOOK CHECKLIST (FINAL)

> **IMPORTANT: This is the last graded notebook. Verify all deliverables.**

- [ ] **4.3.1** LoRA merged + Hybrid RAG integration validated (intent → search → retrieve → generate)
- [ ] **4.4.1** Format Adherence + Exact Match + Fuzzy Match on test split + adversarial subset
- [ ] **4.4.2** Fine-tuning impact quantified (V1 vs V2 with %)
- [ ] **4.5** All 3 architectures scored with SOP-grounded references
- [ ] **`Comparative_Results_Full.csv` saved** ← _FINAL DELIVERABLE_
- [ ] **`Comparative_Results_Summary.csv` saved** ← _FINAL DELIVERABLE_

### Complete Artifact Inventory

| Artifact | Created In |
|---|---|
| `sampled_data.csv` | NB1 |
| `./tokenized_train/`, `./tokenized_valid/`, `df_test.csv` | NB2 |
| `outputs.json` | NB3 + NB4 |
| `./chroma_db/` | NB4 |
| `v1_metrics.csv` | NB5 |
| `./intent_lora_best/`, `training_log.csv`, `training_curves.png` | NB6 |
| `Comparative_Results_Full.csv`, `Comparative_Results_Summary.csv` | NB7 |

**Mark all items checked, then prepare your final submission package.**